> **MIT License** - Copyright (c) 2026 QuantWall. Full text: https://opensource.org/license/mit
>
> **Disclaimer.** This content is general information and software-engineering education only. It is not financial product advice, a recommendation, or an offer to buy or sell any financial product, and it does not take into account your objectives, financial situation or needs. The author does not hold an Australian Financial Services Licence and is not a registered Commodity Trading Advisor. Trading futures and leveraged products carries a high risk of loss. Consult a licensed adviser before making any financial decision.
>
> **HYPOTHETICAL OR SIMULATED PERFORMANCE RESULTS HAVE CERTAIN LIMITATIONS. UNLIKE AN ACTUAL PERFORMANCE RECORD, SIMULATED RESULTS DO NOT REPRESENT ACTUAL TRADING. ALSO, SINCE THE TRADES HAVE NOT BEEN EXECUTED, THE RESULTS MAY HAVE UNDER-OR-OVER COMPENSATED FOR THE IMPACT, IF ANY, OF CERTAIN MARKET FACTORS, SUCH AS LACK OF LIQUIDITY. SIMULATED TRADING PROGRAMS IN GENERAL ARE ALSO SUBJECT TO THE FACT THAT THEY ARE DESIGNED WITH THE BENEFIT OF HINDSIGHT. NO REPRESENTATION IS BEING MADE THAT ANY ACCOUNT WILL OR IS LIKELY TO ACHIEVE PROFIT OR LOSSES SIMILAR TO THOSE SHOWN.**
>
> All data shown is synthetic or publicly available sample data. Features are standard textbook baselines (lagged returns, ATR, RSI, moving averages, volume). No proprietary signal is disclosed or implied.

# The traps: every number in the video

Real: the many-analysts study (Silberzahn et al. 2018), the Linda problem share (Tversky & Kahneman 1983), two US 10-year yield points. Everything else is a seeded simulation on made-up prices with no edge unless stated. Same code as the animations (`animations_manim/traps_sim.py`).

In [ ]:
import numpy as np

import numpy as np

MANY_ANALYSTS = {"teams": 29, "analysts": 61, "significant": 20, "not": 9, "low": 0.89, "high": 2.93, "median": 1.31}
LINDA_SHARE = 0.85
YIELD = {"high_year": 1981, "high": 15.8, "low_year": 2020, "low": 0.5}


def sharpe(r):
    r = np.asarray(r, float)
    return float(r.mean() / r.std() * np.sqrt(252)) if r.std() > 0 else 0.0


# ---------------------------------------------------------------------------------------------
# Overfitting: knobs added one at a time (a calendar rule fit to noise)
# Knob k: sit out the k worst calendar months (by the first ten years). More knobs, better past, same future.
# ---------------------------------------------------------------------------------------------
def knobs(seed=4, years=20, sd=0.01, max_knobs=8, worlds=500):
    def one(s):
        rng = np.random.default_rng(s)
        n = 252 * years
        r = rng.normal(0, sd, n)
        month = (np.arange(n) // 21) % 12
        half = n // 2
        ins, outs = [], []
        order = np.argsort([r[:half][month[:half] == m].mean() for m in range(12)])   # worst months first, in-sample
        for k in range(max_knobs + 1):
            keep = ~np.isin(month, order[:k])
            ins.append(sharpe(np.where(keep, r, 0)[:half]))
            outs.append(sharpe(np.where(keep, r, 0)[half:]))
        return ins, outs, order
    ins, outs, order = one(seed)
    many = np.array([one(1000 + w)[:2] for w in range(worlds)])      # (worlds, 2, knobs)
    return {"k": list(range(max_knobs + 1)), "in": ins, "out": outs, "order": [int(m) for m in order[:max_knobs]],
            "in_avg": many[:, 0].mean(0).tolist(), "out_avg": many[:, 1].mean(0).tolist(), "worlds": worlds}


# ---------------------------------------------------------------------------------------------
# Forking paths: one analysis, but each choice could have depended on the data
# ---------------------------------------------------------------------------------------------
FORKS = [("which market", ["gold", "oil", "wheat"]), ("which years", ["all", "since 2010"]),
         ("which exit", ["next day", "after 5 days"]), ("which filter", ["none", "trend up"])]


def fork_count():
    return int(np.prod([len(o) for _, o in FORKS]))


# ---------------------------------------------------------------------------------------------
# Costs: spreads widen on the days a "big move" rule trades
# Daily volatility clusters (a simple GARCH-like walk); the spread is proportional to that day's volatility;
# the rule trades the day after a big move (the top 10% of absolute returns).
# ---------------------------------------------------------------------------------------------
def spread_paid(seed=2, years=20, base_bp=2.0, top=0.10):
    rng = np.random.default_rng(seed)
    n = 252 * years
    logv = np.zeros(n)
    for t in range(1, n):
        logv[t] = 0.97 * logv[t - 1] + rng.normal(0, 0.12)
    vol = 0.01 * np.exp(logv - logv.mean())
    r = vol * rng.normal(0, 1, n)
    spread = base_bp * vol / vol.mean()
    big = np.abs(r) >= np.quantile(np.abs(r), 1 - top)
    trade_days = np.roll(big, 1)
    trade_days[0] = False
    return {"r": r, "spread": spread, "trade_days": trade_days,
            "avg_bp": float(spread.mean()), "paid_bp": float(spread[trade_days].mean()),
            "ratio": float(spread[trade_days].mean() / spread.mean()), "trades": int(trade_days.sum())}


# ---------------------------------------------------------------------------------------------
# Regime luck: a trend rule in a trending regime, then in a choppy one
# Regime A (15 years): daily returns with positive autocorrelation; regime B (15 years): none.
# Rule: hold the sign of yesterday's return (textbook one-day trend). No other edge.
# ---------------------------------------------------------------------------------------------
def regimes(seed=37, years=15, sd=0.01, rho=0.08, worlds=300):
    def one(s):
        rng = np.random.default_rng(s)
        n = 252 * years
        e = rng.normal(0, sd, 2 * n)
        r = e.copy()
        for t in range(1, n):
            r[t] = rho * r[t - 1] + e[t] * np.sqrt(1 - rho ** 2)
        pos = np.r_[0, np.sign(r[:-1])]
        strat = pos * r
        return sharpe(strat[:n]), sharpe(strat[n:]), strat
    a, b, strat = one(seed)
    many = np.array([one(500 + w)[:2] for w in range(worlds)])
    return {"A": a, "B": b, "strat": strat, "years": years, "A_avg": float(many[:, 0].mean()), "B_avg": float(many[:, 1].mean()),
            "A_pos": float((many[:, 0] > 0).mean()), "B_pos": float((many[:, 1] > 0).mean())}


# ---------------------------------------------------------------------------------------------
# Reactive versions: each version fixes what made the last one fail, on the same data
# Version 1: always in (noise). Each next version sits out the calendar year-type that hurt most last time:
# here, the weekday whose losses were largest in the version's own backtest. Then: ten new years.
# ---------------------------------------------------------------------------------------------
def versions(seed=11, years=10, sd=0.01, n_versions=5, worlds=500):
    def one(s):
        rng = np.random.default_rng(s)
        n = 252 * years
        past, future = rng.normal(0, sd, n), rng.normal(0, sd, n)
        wd = np.arange(n) % 5
        out_days = []
        res = []
        names = ["v1", "v2", "v3", "v4", "v5"][:n_versions]
        for v in range(n_versions):
            keep = ~np.isin(wd, out_days)
            res.append((sharpe(np.where(keep, past, 0)), sharpe(np.where(keep, future, 0))))
            # the failure analysis: which remaining weekday lost the most in this version's backtest?
            left = [d for d in range(5) if d not in out_days]
            worst = min(left, key=lambda d: past[wd == d].sum())
            out_days.append(worst)
        return names, res
    names, res = one(seed)
    many = np.array([one(2000 + w)[1] for w in range(worlds)])   # (worlds, versions, 2)
    return {"names": names, "past": [p for p, _ in res], "future": [f for _, f in res],
            "past_avg": many[:, :, 0].mean(0).tolist(), "future_avg": many[:, :, 1].mean(0).tolist(), "worlds": worlds}

## The hook: many analysts, one data set (real)

In [ ]:
print(MANY_ANALYSTS)

## 1 · Overfitting: knobs on a calendar rule
Each knob sits out the next-worst calendar month, judged on the first ten years. Averages over 500 worlds.

In [ ]:
k = knobs()
for n, a, b in zip(k['k'], k['in_avg'], k['out_avg']):
    print(f'{n} knobs: backtest {a:+.2f}   next ten years {b:+.2f}')

## 2 · Forking paths

In [ ]:
print(FORKS)
print('analyses you could have run:', fork_count())

## 5 · Costs: the spread you pay vs the average spread

In [ ]:
c = spread_paid()
print(f"average spread {c['avg_bp']:.2f} bp; paid by a rule that trades after big moves {c['paid_bp']:.2f} bp ({c['ratio']:.2f}x), {c['trades']} trades")

## 6 · Regime luck: a one-day trend rule in two regimes

In [ ]:
g = regimes()
print(f"trending regime: average Sharpe {g['A_avg']:.2f}, positive in {g['A_pos']:.0%} of worlds")
print(f"no-trend regime: average Sharpe {g['B_avg']:.2f}, positive in {g['B_pos']:.0%} of worlds")
print('real: US 10-year yield', YIELD)

## 7 · The story (real)

In [ ]:
print(f'Linda problem: about {LINDA_SHARE:.0%} chose the conjunction')

## 8 · Reactive versions
Each version sits out the weekday that lost most in the previous version's own backtest. Averages over 500 worlds.

In [ ]:
v = versions()
for n, a, b in zip(v['names'], v['past_avg'], v['future_avg']):
    print(f'{n}: backtest {a:+.2f}   ten new years {b:+.2f}')